In [3]:
# ================================================================
# 🤖 Maharashtra Water Footprint Chatbot (Smart NLP + Sentence-BERT)
# ================================================================

# --- Step 1: Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Step 2: Import dependencies ---
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import re

# --- Step 3: Define paths ---
data_path = "/content/drive/MyDrive/WaterFootprintData/Maharashtra_WaterFootprint_AI.csv"
drive_folder = "/content/drive/MyDrive/Model_training"
os.makedirs(drive_folder, exist_ok=True)

# --- Step 4: Load dataset ---
df = pd.read_csv(data_path)
print("✅ Dataset loaded successfully!\n")
print("Columns:", df.columns.tolist())

# --- Step 5: Create descriptive text for embeddings ---
df["text"] = (
    "The total water footprint of "
    + df["Crop"].astype(str)
    + " in "
    + df["District"].astype(str)
    + " during the "
    + df["Season"].astype(str)
    + " season under "
    + df["Irrigation_Type"].astype(str)
    + " irrigation is "
    + df["Total_Water_Footprint(L_per_kg)"].astype(str)
    + " L/kg, categorized as "
    + df["Category"].astype(str)
    + "."
)

# --- Step 6: Load or train Sentence-BERT model ---
model_path = os.path.join(drive_folder, "maharashtra_water_model")
if os.path.exists(model_path):
    print("📦 Loading saved model from Drive...")
    model = SentenceTransformer(model_path)
else:
    print("🧠 Training Sentence-BERT model (first-time setup)...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    model.save(model_path)
    print(f"✅ Model saved to: {model_path}")

# --- Step 7: Create or load embeddings ---
embedding_path = os.path.join(drive_folder, "maharashtra_water_embeddings.pt")
if os.path.exists(embedding_path):
    print("📂 Loading pre-saved embeddings from Drive...")
    embeddings = torch.load(embedding_path)
else:
    print("⚙️ Generating embeddings (this may take a few minutes)...")
    embeddings = model.encode(df["text"].tolist(), convert_to_tensor=True, show_progress_bar=True)
    torch.save(embeddings, embedding_path)
    print(f"✅ Embeddings saved successfully to: {embedding_path}")

# ================================================================
# --- Step 8: Smart Chatbot Logic ---
# ================================================================

def preprocess_query(q):
    """Normalize text for better semantic matching."""
    q = q.lower().strip()
    q = re.sub(r"[?.!]", "", q)
    q = q.replace("water requirement", "water footprint")
    q = q.replace("usage", "water footprint")
    q = q.replace("consume", "water footprint")
    q = q.replace("require", "water footprint")
    return q

def detect_intent(q):
    """Identify query type (max, min, compare, lookup)."""
    if any(word in q for word in ["highest", "most", "maximum", "uses more", "largest"]):
        return "max"
    elif any(word in q for word in ["lowest", "least", "minimum", "efficient", "saves"]):
        return "min"
    elif "between" in q or "compare" in q:
        return "compare"
    else:
        return "lookup"

def extract_entities(q):
    """Extract crop and district names present in the query."""
    q = q.lower()
    crops = [c for c in df["Crop"].dropna().unique() if c.lower() in q]
    districts = [d for d in df["District"].dropna().unique() if d.lower() in q]
    return crops, districts

def chatbot_reply(user_query):
    """Core chatbot function with intelligent reasoning and fallback."""
    q = preprocess_query(user_query)
    intent = detect_intent(q)
    crops, districts = extract_entities(q)

    # --- Case 1: Specific crop + district lookup ---
    if intent == "lookup" and crops and districts:
        crop, district = crops[0], districts[0]
        subset = df[(df["Crop"] == crop) & (df["District"] == district)]
        if not subset.empty:
            row = subset.iloc[0]
            return (f"🌿 The total water footprint of **{crop}** in **{district}** "
                    f"during the **{row['Season']}** season under **{row['Irrigation_Type']}** irrigation "
                    f"is **{row['Total_Water_Footprint(L_per_kg)']:.2f} L/kg**, "
                    f"categorized as **{row['Category'].lower()}**.")

    # --- Case 2: District-wise max/min query ---
    if intent in ["max", "min"] and districts:
        district = districts[0]
        subset = df[df["District"] == district]
        if not subset.empty:
            row = subset.loc[
                subset["Total_Water_Footprint(L_per_kg)"].idxmax()
            ] if intent == "max" else subset.loc[
                subset["Total_Water_Footprint(L_per_kg)"].idxmin()
            ]
            level = "highest" if intent == "max" else "lowest"
            return (f"💧 In **{district}**, the crop with the **{level} water footprint** "
                    f"is **{row['Crop']}** with **{row['Total_Water_Footprint(L_per_kg)']:.2f} L/kg**, "
                    f"categorized as **{row['Category'].lower()}**.")

    # --- Case 3: Compare crops within same district ---
    if intent == "compare" and len(crops) == 2 and districts:
        district = districts[0]
        subset = df[(df["District"] == district) & (df["Crop"].isin(crops))]
        if len(subset) == 2:
            c1, c2 = subset.iloc[0], subset.iloc[1]
            more = c1 if c1["Total_Water_Footprint(L_per_kg)"] > c2["Total_Water_Footprint(L_per_kg)"] else c2
            less = c1 if c1["Total_Water_Footprint(L_per_kg)"] < c2["Total_Water_Footprint(L_per_kg)"] else c2
            return (f"📊 In **{district}**, **{more['Crop']}** has a higher water footprint "
                    f"({more['Total_Water_Footprint(L_per_kg)']:.2f} L/kg) than **{less['Crop']}** "
                    f"({less['Total_Water_Footprint(L_per_kg)']:.2f} L/kg).")

    # --- Case 4: Semantic similarity fallback (no direct match) ---
    query_emb = model.encode(q, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, embeddings, top_k=3)[0]
    if not hits:
        return "❌ Sorry, I couldn’t find relevant data for your question."

    best = hits[0]
    row = df.iloc[best["corpus_id"]]
    response = (f"🌾 Based on available data, **{row['Crop']}** in **{row['District']}** "
                f"during the **{row['Season']}** season under **{row['Irrigation_Type']}** irrigation "
                f"has a total water footprint of **{row['Total_Water_Footprint(L_per_kg)']:.2f} L/kg**, "
                f"categorized as **{row['Category'].lower()}**.")

    if intent == "max":
        response += " 💧 This crop is considered water-intensive."
    elif intent == "min":
        response += " 🌱 This crop is relatively water-efficient."

    return response


# ================================================================
# --- Step 9: Testing with various query types ---
# ================================================================
test_queries = [
    "What is the water footprint of sugarcane in Pune district?",
    "Which crop has the highest water footprint in Solapur?",
    "Tell me the lowest water consuming crop in Aurangabad.",
    "Give me data about cotton in Kolhapur."
]

print("\n================= 💧 Chatbot Evaluation =================\n")
for i, q in enumerate(test_queries, start=1):
    print(f"🔹 Query {i}: {q}")
    try:
        print("💬 Chatbot Reply:\n", chatbot_reply(q))
    except Exception as e:
        print(f"⚠️ Error: {e}")
    print("------------------------------------------------------")
print("\n✅ All queries processed successfully!\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset loaded successfully!

Columns: ['State', 'District', 'Crop', 'Season', 'Irrigation_Type', 'Total_Water_Footprint(L_per_kg)', 'Category']
📦 Loading saved model from Drive...
📂 Loading pre-saved embeddings from Drive...

================= 💧 Chatbot Evaluation =================

🔹 Query 1: What is the water footprint of sugarcane in Pune district?
💬 Chatbot Reply:
 🌿 The total water footprint of **Sugarcane** in **Pune** during the **Kharif** season under **Drip** irrigation is **1870.01 L/kg**, categorized as **medium**.
------------------------------------------------------
🔹 Query 2: Which crop has the highest water footprint in Solapur?
💬 Chatbot Reply:
 💧 In **Solapur**, the crop with the **highest water footprint** is **Banana** with **4210.84 L/kg**, categorized as **high**.
------------------------------------------------------
🔹 Query 3: Tell 